# 01 — Data Preparation

Part of a multi-notebook pipeline for classical + lightweight-trainable lung segmentation and
detection on the [Chest Xray Masks and Labels](https://www.kaggle.com/datasets/nikhilpandey360/chest-xray-masks-and-labels)
dataset (Montgomery + Shenzhen chest X-rays with lung-field masks).

**Notebook layout:**

1. `01_data_preparation.ipynb` (this notebook) — download, verify, split
2. `02_eda_and_preprocessing.ipynb` — exploratory analysis + preprocessing pipeline
3. `03_segmentation.ipynb` — classical segmentation techniques + benchmarking
4. `04_detection.ipynb` — lightweight trainable detector + benchmarking

Each notebook re-runs top-to-bottom independently; later notebooks load `data/manifest.csv`
produced here rather than re-downloading or re-splitting.

## Step 1 — Data Preparation

### 1.1 Imports and dataset download

We use `kagglehub` to fetch the dataset (assumes Kaggle API credentials are already configured,
e.g. `~/.kaggle/kaggle.json` or `KAGGLE_USERNAME`/`KAGGLE_KEY` env vars). `kagglehub` caches the
download, so re-running this cell on the same machine is a no-op after the first run.

This notebook may be shared publicly, so we never print raw absolute filesystem paths — a small
`anon()` helper masks the home directory as `<USER>` in anything we display. The *real* paths are
still used internally (and saved to the gitignored `data/manifest.csv`) so the pipeline actually
runs; only the printed/displayed text is sanitized.

In [1]:
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import kagglehub

RANDOM_SEED = 42
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

_HOME = str(Path.home())

def anon(path) -> str:
    # mask the local home directory in a path before printing/displaying it
    return str(path).replace(_HOME, "<USER>")

dataset_path = Path(kagglehub.dataset_download("nikhilpandey360/chest-xray-masks-and-labels"))
print("kagglehub cache path:", anon(dataset_path))

kagglehub cache path: <USER>\.cache\kagglehub\datasets\nikhilpandey360\chest-xray-masks-and-labels\versions\1


### 1.2 Locate the actual data folder

The downloaded archive contains the real data under a `Lung Segmentation/` folder, plus a
byte-for-byte duplicate of the same tree nested under `data/Lung Segmentation/` (an artifact of
how the archive was packaged upstream). We only need one copy, so we point at the top-level one
and search for it defensively in case the layout differs on another machine (e.g. Colab).

In [2]:
root_candidates = [dataset_path / "Lung Segmentation"]
if not root_candidates[0].exists():
    root_candidates = list(dataset_path.rglob("CXR_png"))
    root_candidates = [p.parent for p in root_candidates]

assert root_candidates, f"Could not locate the 'Lung Segmentation' data folder under {anon(dataset_path)}"
ROOT = root_candidates[0]
print("Using data root:", anon(ROOT))

CXR_DIR = ROOT / "CXR_png"
MASK_DIR = ROOT / "masks"
TEST_DIR = ROOT / "test"  # Shenzhen images released WITHOUT ground-truth masks (Kaggle held-out test)

for d in [CXR_DIR, MASK_DIR, TEST_DIR]:
    print(f"{d.name:10s} exists={d.exists()}  n_files={len(list(d.glob('*.png')))}")

Using data root: <USER>\.cache\kagglehub\datasets\nikhilpandey360\chest-xray-masks-and-labels\versions\1\Lung Segmentation
CXR_png    exists=True  n_files=800
masks      exists=True  n_files=704
test       exists=True  n_files=96


### 1.3 Build the manifest and verify image–mask correspondence

Filenames encode the source dataset:
- `MCUCXR_xxxx_y.png` → Montgomery County (MC) set
- `CHNCXR_xxxx_y.png` → Shenzhen (China, CHN) set

Each mask is named `<image_stem>_mask.png`. We match every mask back to its image, and explicitly
flag (rather than silently ignore) any mask with no corresponding image, and any image that is
neither matched to a mask nor present in the unlabeled `test/` folder.

In [3]:
def source_from_name(name: str) -> str:
    if name.startswith("MCUCXR"):
        return "Montgomery"
    if name.startswith("CHNCXR"):
        return "Shenzhen"
    return "Unknown"

cxr_files = sorted(CXR_DIR.glob("*.png"))
mask_files = sorted(MASK_DIR.glob("*.png"))
test_only_files = sorted(TEST_DIR.glob("*.png"))

cxr_names = {f.name: f for f in cxr_files}

mask_by_image_name = {}
unmatched_masks = []
for mp in mask_files:
    img_name = mp.name.replace("_mask.png", ".png")
    if img_name in cxr_names:
        mask_by_image_name[img_name] = mp
    else:
        unmatched_masks.append(mp.name)  # mask with no corresponding image -> dropped

matched_names = set(mask_by_image_name.keys())
test_names = set(f.name for f in test_only_files)
orphan_images = set(cxr_names.keys()) - matched_names - test_names  # image with neither mask nor test/ -> dropped

print(f"Masks total:                                   {len(mask_files)}")
print(f"Masks matched to an image:                     {len(matched_names)}")
print(f"Masks with NO corresponding image (dropped):   {len(unmatched_masks)}  {unmatched_masks[:10]}")
print(f"Images with neither mask nor test/ (dropped):  {len(orphan_images)}  {list(orphan_images)[:10]}")
print(f"Unlabeled images held out in test/ (no mask):  {len(test_only_files)}")

records = [
    {
        "id": name.replace(".png", ""),
        "source": source_from_name(name),
        "image_path": str(cxr_names[name]),
        "mask_path": str(mask_by_image_name[name]),
    }
    for name in sorted(matched_names)
]
manifest = pd.DataFrame.from_records(records)
print(f"\nFinal labeled manifest size: {len(manifest)} image/mask pairs")
manifest["source"].value_counts()

Masks total:                                   704
Masks matched to an image:                     704
Masks with NO corresponding image (dropped):   0  []
Images with neither mask nor test/ (dropped):  0  []
Unlabeled images held out in test/ (no mask):  96

Final labeled manifest size: 704 image/mask pairs


source
Shenzhen      566
Montgomery    138
Name: count, dtype: int64

### 1.4 Train / Val / Test split (70/15/15, stratified by source)

We split only the 704 labeled pairs (image+mask). The 96 unlabeled Shenzhen images in `test/`
have no ground truth, so they can't be used for quantitative segmentation/detection benchmarking —
we keep them listed separately for optional qualitative inference later, but they are excluded
from the train/val/test split.

The split is stratified by `source` (Montgomery vs Shenzhen) so both datasets are proportionally
represented in every split, since the two sources differ noticeably in resolution and acquisition
characteristics (see Step 1.5). Note: `image_path`/`mask_path` in the saved manifest are absolute
local paths (needed so later notebooks can actually load the files) — the manifest CSV is
gitignored for this reason and is regenerated deterministically (fixed seed) by this notebook.

In [4]:
train_df, temp_df = train_test_split(
    manifest, test_size=0.30, stratify=manifest["source"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["source"], random_state=RANDOM_SEED
)

train_df = train_df.copy(); train_df["split"] = "train"
val_df = val_df.copy();     val_df["split"] = "val"
test_df = test_df.copy();   test_df["split"] = "test"

full_manifest = pd.concat([train_df, val_df, test_df]).sort_values("id").reset_index(drop=True)

manifest_path = DATA_DIR / "manifest.csv"
full_manifest.to_csv(manifest_path, index=False)
print(f"Saved manifest with split assignments -> {manifest_path}")

pd.crosstab(full_manifest["split"], full_manifest["source"], margins=True, margins_name="Total")

Saved manifest with split assignments -> ..\data\manifest.csv


source,Montgomery,Shenzhen,Total
split,,,
test,21,85,106
train,96,396,492
val,21,85,106
Total,138,566,704


### 1.5 Dataset report

Size, image dimensions/formats, and lung-vs-background class balance, broken down by source.
Class balance is computed as the fraction of foreground (lung) pixels per mask using each PNG's
color histogram (fast — avoids decoding full-resolution arrays for ~700 images, some of which are
~20 megapixels).

In [5]:
def image_size(path):
    with Image.open(path) as im:
        return im.size, im.mode  # (width, height), mode

def foreground_ratio(mask_path):
    with Image.open(mask_path) as im:
        im = im.convert("L")
        hist = im.histogram()  # 256-bin grayscale histogram, cheap (no full array copy)
    total = sum(hist)
    fg = sum(hist[128:])  # masks are binary {0, 255}; >=128 counts as lung
    return fg / total

rows = []
for r in full_manifest.itertuples():
    (w, h), mode = image_size(r.image_path)
    fg_ratio = foreground_ratio(r.mask_path)
    rows.append({"id": r.id, "source": r.source, "split": r.split,
                 "width": w, "height": h, "mode": mode, "lung_pixel_ratio": fg_ratio})

stats_df = pd.DataFrame(rows)
stats_df.to_csv(DATA_DIR / "manifest_stats.csv", index=False)

print("=== Dataset size ===")
print(f"Total labeled pairs: {len(full_manifest)}  "
      f"(Montgomery: {(full_manifest.source=='Montgomery').sum()}, "
      f"Shenzhen: {(full_manifest.source=='Shenzhen').sum()})")
print(f"Unlabeled Shenzhen images (excluded, no ground truth): {len(test_only_files)}")

print("\n=== Image dimensions by source ===")
print(stats_df.groupby("source")[["width", "height"]].agg(["min", "max", "mean"]).round(1))
print(f"\nDistinct (width, height) combinations: {stats_df.groupby(['width','height']).ngroups}")

print("\n=== PIL image mode (color format) by source ===")
print(pd.crosstab(stats_df["source"], stats_df["mode"]))

print("\n=== Lung vs background pixel ratio (class balance) ===")
print(stats_df.groupby("source")["lung_pixel_ratio"].agg(["mean", "std", "min", "max"]).round(4))
print("\nOverall:")
print(stats_df["lung_pixel_ratio"].agg(["mean", "std", "min", "max"]).round(4))

=== Dataset size ===
Total labeled pairs: 704  (Montgomery: 138, Shenzhen: 566)
Unlabeled Shenzhen images (excluded, no ground truth): 96

=== Image dimensions by source ===
           width               height              
             min   max    mean    min   max    mean
source                                             
Montgomery  4020  4892  4279.1   4020  4892  4632.9
Shenzhen    1225  3001  2702.3    989  3001  2797.1

Distinct (width, height) combinations: 455

=== PIL image mode (color format) by source ===
mode          L    P  RGB
source                   
Montgomery  138    0    0
Shenzhen      0  539   27

=== Lung vs background pixel ratio (class balance) ===
              mean     std     min     max
source                                    
Montgomery  0.2555  0.0616  0.0988  0.3977
Shenzhen    0.2526  0.0576  0.0835  0.4326

Overall:
mean    0.2532
std     0.0584
min     0.0835
max     0.4326
Name: lung_pixel_ratio, dtype: float64
